Phase 1 (Ingestion)

In [0]:
# Define Schemas
# -------------------------------

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

# Customers
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("country", StringType(), True)
])

# Orders
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("total_amount", DoubleType(), True)
])

# Products
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("stock_quantity", IntegerType(), True)
])


In [0]:

#  Read Data using Autoloader from RAW

customers_df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    #.option("cloudFiles.inferSchema", "true")
    .option("header", "true")
    .schema(customers_schema)
    .load("/Volumes/ecom/storage/project/raw/customers/")
)

products_df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    #.option("cloudFiles.inferSchema", "true")
    .schema(products_schema)
    .load("/Volumes/ecom/storage/project/raw/products/")
)

orders_df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    #.option("cloudFiles.inferSchema", "true")
    .schema(orders_schema)
    .load("/Volumes/ecom/storage/project/raw/orders/")
)

In [0]:

#  Write to Bronze Delta Tables

customers_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecom/storage/project/checkpoints/customers/") \
    .trigger(once=True) \
    .start("/Volumes/ecom/storage/project/bronze/customers/")

products_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecom/storage/project/checkpoints/products/") \
    .trigger(once=True) \
    .start("/Volumes/ecom/storage/project/bronze/products/")

orders_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecom/storage/project/checkpoints/orders/") \
    .trigger(once=True) \
    .start("/Volumes/ecom/storage/project/bronze/orders/")

Phase 2 (Bronze to Silver)

In [0]:
# Read Bronze Delta Tables as Stream

customers_stream = spark.readStream.format("delta") \
    .load("/Volumes/ecom/storage/project/bronze/customers/")

products_stream = spark.readStream.format("delta") \
    .load("/Volumes/ecom/storage/project/bronze/products/")

orders_stream = spark.readStream.format("delta") \
    .load("/Volumes/ecom/storage/project/bronze/orders")

In [0]:
from pyspark.sql.functions import col, to_date, current_timestamp

# -------------------------------
# 1️⃣ Clean Customers
# -------------------------------
clean_customers = customers_stream \
    .dropna(subset=["customer_id", "signup_date"]) \
    .na.fill({
        "first_name": "Unknown",
        "last_name": "Unknown",
        "email": "unknown@example.com"
    }) \
    .withColumn("signup_date", to_date(col("signup_date"), "yyyy-MM-dd")) \
    .withColumn("ingestion_date", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .dropDuplicates([
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "country",
        "signup_date"
    ])

# -------------------------------
# 2️⃣ Clean Products
# -------------------------------
clean_products = products_stream \
    .dropna(subset=["product_id"]) \
    .na.fill({
        "product_name": "Unknown",
        "category": "Misc",
        "price": 0.0
    }) \
    .filter(col("price") >= 0) \
    .withColumn("ingestion_date", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .dropDuplicates([
        "product_id",
        "product_name",
        "category",
        "price"
    ])

# -------------------------------
# 3️⃣ Clean Orders
# -------------------------------
clean_orders = orders_stream \
    .dropna(subset=["order_id", "customer_id", "product_id", "order_date"]) \
    .na.fill({
        "quantity": 1,
        "total_amount": 0.0
    }) \
    .filter((col("quantity") > 0) & (col("total_amount") >= 0)) \
    .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd")) \
    .withColumn("ingestion_date", current_timestamp()) \
    .withColumn("source_file", col("_metadata.file_path")) \
    .dropDuplicates([
        "order_id",
        "customer_id",
        "product_id",
        "order_date",
        "quantity",
        "total_amount"
    ])


In [0]:

#  Write Cleaned Data to Silver Delta (Streaming)


clean_customers.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecom/storage/project/checkpoints/customers_silver/") \
    .partitionBy("signup_date") \
    .trigger(availableNow=True)\
    .start("/Volumes/ecom/storage/project/silver/customers/")

clean_products.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True)\
    .option("checkpointLocation", "/Volumes/ecom/storage/project/checkpoints/products_silver/") \
    .start("/Volumes/ecom/storage/project/silver/products/")

clean_orders.writeStream \
    .format("delta") \
    .outputMode("append") \
    .trigger(availableNow=True)\
    .option("checkpointLocation", "/Volumes/ecom/storage/project/checkpoints/orders_silver/") \
    .partitionBy("order_date") \
    .start("/Volumes/ecom/storage/project/silver/orders/")

Phase 3 (Silver to Gold)

In [0]:
# Reading silver
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp

# Silver table path
silver_products_path = "/Volumes/ecom/storage/project/silver/products/"

products_silver = spark.read.format("delta").load(silver_products_path)

#reference
gold_products_table = DeltaTable.forName(spark, "gold.dim_products")


# SCD TYPE 1 Merge
gold_products_table.alias("gold").merge(
    products_silver.alias("silver"),
    "gold.product_id = silver.product_id"
).whenMatchedUpdate(set={
    "product_name": col("silver.product_name"),
    "category": col("silver.category"),
    "price": col("silver.price"),
    "ingestion_date": current_timestamp()
}).whenNotMatchedInsert(values={
    "product_id": col("silver.product_id"),
    "product_name": col("silver.product_name"),
    "category": col("silver.category"),
    "price": col("silver.price"),
    "ingestion_date": current_timestamp()
}).execute()

In [0]:
# Reading silver
from delta.tables import DeltaTable
from pyspark.sql.functions import col, current_timestamp

silver_customers_path = "/Volumes/ecom/storage/project/silver/customers/"

customers_silver = spark.read.format("delta").load(silver_customers_path)

#reference
gold_customers_table = DeltaTable.forName(spark, "gold.dim_customers")


# SCD TYPE 2 
from pyspark.sql.functions import lit, current_timestamp, col

gold_customers_table.alias("gold").merge(
    customers_silver.alias("silver"),
    "gold.customer_id = silver.customer_id AND gold.is_current = True"
).whenMatchedUpdate(
    condition=(
        "gold.first_name <> silver.first_name OR "
        "gold.last_name <> silver.last_name OR "
        "gold.email <> silver.email OR "
        "gold.country <> silver.country"
    ),
    set={
        "is_current": lit(False),
        "effective_to": current_timestamp()
    }
).whenNotMatchedInsert(
    values={
        "customer_id": col("silver.customer_id"),
        "first_name": col("silver.first_name"),
        "last_name": col("silver.last_name"),
        "email": col("silver.email"),
        "signup_date": col("silver.signup_date"),
        "country": col("silver.country"),
        "effective_from": current_timestamp(),
        "effective_to": lit(None),
        "is_current": lit(True),
        "ingestion_date": current_timestamp()
    }
).execute()

In [0]:
%sql
--optimizing & z-ordering

OPTIMIZE gold.dim_customers
ZORDER BY (customer_id);

OPTIMIZE gold.dim_products
ZORDER BY (product_id, category);

Phase 4 & 5  (Modeling, Performance , Optimization)

In [0]:
from pyspark.sql.functions import date_format, col

# read silver orders
orders = spark.read.format("delta") \
    .load("/Volumes/ecom/storage/project/silver/orders/")

# add date_key
orders = orders.withColumn(
    "date_key",
    date_format(col("order_date"), "yyyyMMdd").cast("int")
)

# select fact columns
fact_orders = orders.select(
    "order_id",
    "customer_id",
    "product_id",
    "date_key",
    "quantity",
    "total_amount"
)


# write fact table to gold
fact_orders.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold.fact_orders")

Fact Table is partitioned by date_key

In [0]:
%sql
-- Optimize
OPTIMIZE gold.fact_orders;

-- Z-Order
OPTIMIZE gold.fact_orders
ZORDER BY (customer_id, product_id);

-- Vacuum the table
VACUUM gold.fact_orders RETAIN 168 HOURS;
VACUUM gold.dim_customers RETAIN 168 HOURS;
VACUUM gold.dim_products RETAIN 168 HOURS;


In [0]:
%sql
--Collects column statistics so the query engine can choose the best plan:
-- Collect statistics for fact_orders
ANALYZE TABLE gold.fact_orders COMPUTE STATISTICS;

-- For dimensions
ANALYZE TABLE gold.dim_customers COMPUTE STATISTICS;
ANALYZE TABLE gold.dim_products COMPUTE STATISTICS;
ANALYZE TABLE gold.dim_date COMPUTE STATISTICS;